# Ensemble Methods
We are going to be testing ensemble methods for predicting stock prices

## ESNForest MSFT Experiment

Train an ESNForest ensemble on MSFT time-series features generated by `make_single_stock_df`.

In [1]:
# Imports and notebook setup
from pathlib import Path
import os
import sys

import torch
from torch.utils.data import TensorDataset

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from my_engine.config import ModelConfig, TrainerConfig, MetricsConfig
from my_engine.data import get_dataloaders as get_data_loaders
from my_engine.financial_data import make_single_stock_df
from my_engine.trainer import Trainer
from my_engine.utils import build_model, make_optimizer, get_direction_accuracy

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
DEVICE

device(type='cuda')

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Load MSFT data and convert targets to column vectors for regression
WINDOW_SIZE = 30

train_ds, val_ds, test_ds, scaler, features = make_single_stock_df(
    ticker="MSFT",
    period="5y",
    train_split=0.8,
    val_split=0.1,
    window_size=WINDOW_SIZE,
)


def with_column_targets(dataset):
    X, y = dataset.tensors
    if y.ndim == 1:
        y = y.unsqueeze(-1)
    return TensorDataset(X, y)


train_ds = with_column_targets(train_ds)
val_ds = with_column_targets(val_ds)
test_ds = with_column_targets(test_ds) if test_ds is not None else None

num_inputs = len(features)
print(f"features ({num_inputs}): {features}")
print(f"train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds) if test_ds is not None else 0}")

[*********************100%***********************]  1 of 1 completed


features (11): ['log_returns', 'log_volume', 'log_intraday_chng', '10_log_returns_ma', '20_log_returns_ma', '50_log_returns_ma', 'open_close', 'overnight_gap', 'ret_1', 'ret_5', 'variance']
train=933, val=210, test=90


In [4]:
# Dataloaders for time-series training
trainer_config = TrainerConfig(
    optimizer_name="adam",
    learning_rate=1e-3,
    weight_decay=1e-1,
    device=DEVICE,
    num_epochs=10,
    trainer_batch_size=64,
    evaluator_batch_size=128,
    early_stopping_patience=5,
    checkpoint_dir="checkpoints/esn_forest_msft",
)

train_loader, val_loader, test_loader = get_data_loaders(
    train_ds=train_ds,
    eval_ds=val_ds,
    test_ds=test_ds,
    train_batch_size=trainer_config.trainer_batch_size,
    eval_batch_size=trainer_config.evaluator_batch_size,
    test_batch_size=trainer_config.evaluator_batch_size,
    time_series=True,
)

In [11]:
# Build ESNForest from ModelConfig and build_model
esn_forest_config = ModelConfig(
    model_type="esn_forest",
    hidden_units=[128],
    dropout=[0.2],
    resevior_sizes=[25, 50 , 75, 100, 200, 250],
    esn_depths=[1, 2, 3, 4],
    leak_rate_range=(0.2, 1.0),
    reservoir_sparsity_range=(0.5, 0.95),
    spectral_radius_range=(0.5, 1.2),
    input_scale_range=(0.1, 0.8),
    number_esns=100,
)

esn_forest_model = build_model(
    input_spec=num_inputs,
    num_outputs=1,
    config=esn_forest_config,
)

print(esn_forest_model)
print("parameters:", esn_forest_model.num_parameters())
for idx, member_config in enumerate(esn_forest_model.member_configs):
    print(
        f"member {idx}: type={member_config.model_type}, "
        f"reservoir={member_config.reservoir_size}, "
        f"depth={member_config.rnn_num_layers}, "
        f"leak={member_config.leak_rate:.3f}, "
        f"sparsity={member_config.reservoir_sparsity:.3f}, "
        f"radius={member_config.spectral_radius:.3f}, "
        f"input_scale={member_config.input_scale:.3f}"
    )

ESNForest(input=11, members=100, out=1)
parameters: (21590600, 1990500)
member 0: type=deep_esn, reservoir=100, depth=3, leak=0.609, sparsity=0.557, radius=1.195, input_scale=0.641
member 1: type=deep_esn, reservoir=50, depth=4, leak=0.972, sparsity=0.869, radius=0.502, input_scale=0.425
member 2: type=deep_esn, reservoir=50, depth=3, leak=0.976, sparsity=0.623, radius=0.514, input_scale=0.590
member 3: type=deep_esn, reservoir=75, depth=4, leak=0.611, sparsity=0.685, radius=1.034, input_scale=0.710
member 4: type=deep_esn, reservoir=25, depth=4, leak=0.913, sparsity=0.839, radius=1.023, input_scale=0.453
member 5: type=deep_esn, reservoir=100, depth=4, leak=0.829, sparsity=0.828, radius=0.668, input_scale=0.245
member 6: type=esn, reservoir=100, depth=1, leak=0.851, sparsity=0.748, radius=1.045, input_scale=0.743
member 7: type=esn, reservoir=25, depth=1, leak=0.792, sparsity=0.584, radius=0.619, input_scale=0.409
member 8: type=deep_esn, reservoir=100, depth=4, leak=0.856, sparsity=0

In [12]:
# Train ESNForest with TrainerConfig and Trainer
optimizer = make_optimizer(
    filter(lambda p: p.requires_grad, esn_forest_model.parameters()),
    trainer_config,
)
criterion = torch.nn.MSELoss()

with Trainer(
    model=esn_forest_model,
    optimizer=optimizer,
    criterion=criterion,
    config=trainer_config,
    metrics_config=MetricsConfig(task="regression", names=["mse", "mae"])
) as trainer:
    esn_forest_results = trainer.fit(train_loader, val_loader)

esn_forest_results

Epoch 0: Train Loss=1.0249,Val Loss=0.8321, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError': tensor(1.0249, device='cuda:0'), 'MeanAbsoluteError': tensor(0.7394, device='cuda:0')}, Test: {'MeanSquaredError': tensor(0.8321, device='cuda:0'), 'MeanAbsoluteError': tensor(0.6233, device='cuda:0')}
--> Saving checkpoint: checkpoints/esn_forest_msft/last.pt
--> New best checkpoint saved: checkpoints/esn_forest_msft/best.pt
--> Also saving as last checkpoint: checkpoints/esn_forest_msft/last.pt
Epoch 1: Train Loss=1.0245,Val Loss=0.8327, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError': tensor(1.0245, device='cuda:0'), 'MeanAbsoluteError': tensor(0.7389, device='cuda:0')}, Test: {'MeanSquaredError': tensor(0.8327, device='cuda:0'), 'MeanAbsoluteError': tensor(0.6233, device='cuda:0')}
--> Saving checkpoint: checkpoints/esn_forest_msft/last.pt
Epoch 2: Train Loss=1.0239,Val Loss=0.8327, Val Acc=0.00%,Metrics: Train: {'MeanSquaredError': tensor(1.0239, device='cuda:0'), 'MeanAbsoluteError': 

{'num_epochs': 6,
 'train_loss': 1.0239227855576067,
 'train_acc': 0.0,
 'val_loss': 0.8322915133975801,
 'val_acc': 0.0,
 'val_losses': [0.8320524567649478,
  0.8327100867316837,
  0.8327331656501407,
  0.8325321356455485,
  0.832375378835769,
  0.8322915133975801],
 'train_losses': [1.0249246283487237,
  1.0244995221447715,
  1.0239274059716783,
  1.0238950832608174,
  1.0238537951936548,
  1.0239227855576067],
 'train_accs': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 'val_accs': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 'MeanSquaredError': tensor(0.8323, device='cuda:0'),
 'MeanAbsoluteError': tensor(0.6228, device='cuda:0')}

In [7]:
# Evaluate on the held-out MSFT test split
if test_loader is not None:
    test_loss, _, _ = trainer.validate(test_loader)
    print(f"ESNForest MSFT test MSE: {test_loss:.6f}")
else:
    print("No test split was created.")

ESNForest MSFT test MSE: 1.364580


In [14]:
dir_acc, true_dirs, pred_dirs = get_direction_accuracy(esn_forest_model, val_loader, scaler, log_returns_idx=0)
print(f"Directional Accuracy: {dir_acc:.2%}")

Directional Accuracy: 49.52%
